# 第2章：OpenMP SpMV

> 实验主入口：[OpenMP SpMV 实验手册](EXPERIMENT_GUIDE.md)。请按目标、核心代码、编译、运行、正确性、性能和练习的顺序完成并填写结果表。

本章从串行 CSR SpMV 出发，使用 OpenMP 在共享地址空间中按行并行，并用正确性和加速比分析多核扩展性。课程工程保留原项目的 CMake/OpenMP 构建路径，并由构建脚本明确选择鲲鹏毕昇 Host 编译器 `clang++`。

## 前置要求

- 完成第1章
- 理解数组和循环
- 具备可用的 OpenMP 主机编译环境

## 本章学习目标

- 解释 CSR 三数组和 SpMV 访问模式
- 识别 `parallel for schedule(static)` 的任务划分
- 控制线程数并分析内存带宽限制

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
import os, platform, shutil, subprocess
print("Python:", platform.python_version())
print("Platform:", platform.platform())
print("CMake:", shutil.which("cmake"))
cxx = shutil.which("clang++")
print("BiSheng clang++:", cxx or "not found")
if cxx:
    output = subprocess.run([cxx, "--version"], check=False, capture_output=True, text=True).stdout
    print(output.splitlines()[0] if output else "version unavailable")
print("OMP_NUM_THREADS:", os.getenv("OMP_NUM_THREADS", "not set"))


## 章节内容

- [02.01_chapter_intro](02.01_chapter_intro.ipynb)：章节目标和共享内存并行定位
- [02.02_csr_and_spmv_basics](02.02_csr_and_spmv_basics.ipynb)：CSR 与串行 SpMV
- [02.03_openmp_programming_model](02.03_openmp_programming_model.ipynb)：parallel for、schedule 与线程
- [02.04_openmp_spmv_implementation](02.04_openmp_spmv_implementation.ipynb)：构建和运行真实工程子集
- [02.05_scaling_and_bandwidth_analysis](02.05_scaling_and_bandwidth_analysis.ipynb)：加速比与内存带宽
- [02.06_chapter_test](02.06_chapter_test.ipynb)：线程和调度单变量实验

## 实验材料说明

本章完整工程位于当前章节的 `src/`，练习参考位于 `answer/`。Notebook 使用相对路径访问材料，不依赖开发者本机的原始工程位置。

## 预期现象与结果分析

上面的目录检查应显示本章 Notebook、`answer`、`images` 和 `src`。若文件缺失，应先检查课程检出是否完整，而不是继续执行后续实验。按目录顺序学习，先确认正确性，再记录性能；不要用未启用真实后端的 stub 或 reference 路径代表 NPU 性能。

## 章节小结

本节给出了本章的能力目标、材料入口和学习顺序。下一节开始进入具体知识与实验。

## 实验工程说明与本章任务

本章实验工程位于 `src/openmp_spmv/`。课程公共目录 `../common/sparse/` 提供 CSR 数据结构、矩阵生成与串行 CPU 参考实现；`src/openmp_spmv/src/cpu_omp_spmv.cpp` 是 OpenMP 实现；`src/openmp_spmv/benchmark/spmv_benchmark.cpp` 负责计时与结果校验；脚本完成构建运行，CSV 保存历史参考。

### 本章实验任务

构建 → 跑串行/OpenMP 基线 → 设置 `OMP_NUM_THREADS=1,2,4,8,16` → 设置 `OMP_SCHEDULE=static,dynamic,guided` → 固定输入与 repeat → 记录并分析。

所有路径均相对 Notebook 当前目录。先检查环境，再运行真实工程；历史结果只用于观察趋势。

### 完成标准

能够指出核心源码和脚本职责，完成可用环境内的构建/诊断，并按“Threads、Schedule、CPU single、OpenMP、Speedup、Error”记录证据。
